# MCP Tool Use with Cohere

This notebook shows how to use the Cohere `command-a-03-2025` model with tools exposed by a
remote MCP server — using the standard [`mcp`](https://pypi.org/project/mcp/) Python SDK.

We connect to [TWZRD Agent Intel](https://intel.twzrd.xyz), a live production MCP server that
provides trust scoring for Web3 AI agents. The free tools work without any API key, making
this example runnable out-of-the-box alongside your `COHERE_API_KEY`.

**Tools used:**
- `score_agent(wallet)` — returns a 0–100 trust score + risk signals for a wallet address
- `preflight_check(wallet)` — quick go/no-go check before making an x402 payment

**MCP config** (for Claude Desktop):
```json
{"mcpServers": {"twzrd-agent-intel": {"url": "https://intel.twzrd.xyz/mcp"}}}
```

In [ ]:
# Install dependencies
# !pip install cohere mcp

In [ ]:
import asyncio
import json
import os

import cohere
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

COHERE_API_KEY = os.environ.get("COHERE_API_KEY", "<YOUR_COHERE_API_KEY>")
TWZRD_MCP_URL = "https://intel.twzrd.xyz/mcp"
MODEL = "command-a-03-2025"

# Wallet to check — a known repeat x402 payer from the Dexter ecosystem
WALLET = "D1QkbFJKiPsymJ65RKHhF6DFB8sPMfpBaFBzuHKfJGWi"

In [ ]:
async def get_mcp_tools():
    """Fetch tool schemas from the TWZRD MCP server and convert to Cohere tool format."""
    async with streamablehttp_client(TWZRD_MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_result = await session.list_tools()

            cohere_tools = []
            for tool in tools_result.tools:
                cohere_tool = {
                    "name": tool.name,
                    "description": tool.description or "",
                    "parameter_definitions": {
                        k: {
                            "description": v.get("description", ""),
                            "type": v.get("type", "str"),
                            "required": k in tool.inputSchema.get("required", []),
                        }
                        for k, v in tool.inputSchema.get("properties", {}).items()
                    },
                }
                cohere_tools.append(cohere_tool)
    return cohere_tools

tools = asyncio.run(get_mcp_tools())
print(f"Loaded {len(tools)} tools from TWZRD MCP server:")
for t in tools:
    print(f"  - {t['name']}: {t['description'][:60]}...")

In [ ]:
async def call_mcp_tool(tool_name: str, tool_input: dict) -> str:
    """Execute a tool call against the TWZRD MCP server."""
    async with streamablehttp_client(TWZRD_MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, tool_input)
            return result.content[0].text


def run_agent(question: str, wallet: str) -> str:
    """Agentic loop: Cohere decides which MCP tools to call, we execute them, loop until done."""
    co = cohere.ClientV2(api_key=COHERE_API_KEY)

    messages = [{"role": "user", "content": f"{question} Wallet: {wallet}"}]
    tool_results = []

    while True:
        response = co.chat(
            model=MODEL,
            messages=messages,
            tools=tools,
        )

        # No tool calls → final answer
        if not response.message.tool_calls:
            return response.message.content[0].text

        # Execute each tool call
        messages.append(response.message)
        tool_results = []
        for tc in response.message.tool_calls:
            print(f"  → Calling {tc.function.name}({tc.function.arguments})")
            result = asyncio.run(
                call_mcp_tool(tc.function.name, json.loads(tc.function.arguments))
            )
            tool_results.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

        messages.extend(tool_results)

In [ ]:
question = "Should I make an x402 payment to this Web3 agent wallet? Check its trust score and preflight status, then give a recommendation."
print(f"Question: {question}\n"  )
answer = run_agent(question, WALLET)
print(f"\nFinal answer:\n{answer}")